In [1]:
%%html
<style type="text/css">
  span.ecb { background: yellow; }
</style>
<span class="ecb">Grading comments will be in yellow</span>

In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from hw5_eqs import *

# ATMO 5322 - Homework #4
### Spring 2026, Due 5 May, 2026, 2 pm (before class).

This homework builds on Homework 4 to calculate 

For homework 4 and 5, you my work individually or with a group of up to two other class members. If you work together, please say who you worked with in the assignment and only turn in one assignment on GitHub.

When working these problems, you will need to code up equations for and make plots of various electrostatic quantities. Turn in a Jupyter notebook with your figures and which I can run to reproduce your results. Label your axes like a professional.

Please also enter your derivations into the notebook using $\LaTeX$. Provide any needed discussion as commentary in the notebook in a Markdown cell.

To the extent possible, move supporting calculation code to another file and import it so the notebook is shorter and cleaner.

The next set of questions will scale up the calculation of electric field change and its inversion using the code from the previous part of this assignment. This will allow us to study the errors in retrieving charges and their locations starting from known input data.

### Q1: Generating dE from known monopolar discharges

To study the sensitivity of solution quality to its position within the field change network, we will next move the monopolar total charge amount from stroke 6a to many different locations. Use a 1 km grid covering (-10, 10) in x and (-15, 15) in y, and assume the charge is lowered from 3, 5, and 7 km above ground. Predict the field change at each station (you don't need to plot it).

This calculation can be vectorized for all grid locations and one site. In my implementation the calculation of dE for all grid boxes for all eight sites runs almost instantly.

### Q2: Inferring the monopolar discharge from the generated dE

You now have dE at all station locations, for all of the (assumed) monopolar discharges. Retreive the locations and charge amounts, pretending you didn't know them. This calculation will be slower, but only required a few seconds to completely run in my implementation.

Calculate and plot the error in the source location and charge magnitude at each position (a total of four error variables). Also plot the location of the measurement statios. There should be twelve plots - sets of four at each altitude. Is there any spatial variaibility in these error measures?

### Q3. Effect of measurement error on retrievals

Add a zero-mean distribution of [Gaussian errors](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html#numpy.random.Generator.normal) to the measured field changes. Use a standard deviation (aka RMS error) of 150 V/m, independently distributed at each station. 

Repeat the retrieval process and plot diagnostics for each error corrupted measurement. 

### Q4. Statistics of retrieval errors

Repeat Q3 for 30 error-corrupted E field measurements, so as to produce a large sample of statistics of the errors.

Calculate and plot the bias and standard deviation of position (each coordinate) and charge magnitude at each charge source location. This will be 24 plots, using all 30 realizations of the retrieval at each grid point.

### Q5: Visualizing the error stats as a chi^2 distribution
Calculate the reduced chi-squared value (in V/m) associated with each retrieved source, accounting for the degrees of freedom $\nu$, and normalizing by the RMS error you added above. You will have about 54000 values = (20x30x3 spatial)x(30 retrievals, one for each realization of error at each station). Create a histogram of these chi-squared values, and compare to the [analytic chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) for that $\nu$, as in one of the panels in Thomas et al. (2004), Figure 9.

### Q1: Generating dE from known monopolar discharges

To study the sensitivity of solution quality to its position within the field change network, we will next move the monopolar total charge amount from stroke 6a to many different locations. Use a 1 km grid covering (-10, 10) in x and (-15, 15) in y, and assume the charge is lowered from 3, 5, and 7 km above ground. Predict the field change at each station (you don't need to plot it).

This calculation can be vectorized for all grid locations and one site. In my implementation the calculation of dE for all grid boxes for all eight sites runs almost instantly.

In [3]:
# station df from sam
station_df = pd.DataFrame([
    {'name' : 'STROZZI', 'x' : -200, 'y' : 100, 'z' : 0},
    {'name' : 'RECEIVING VANS', 'x' : -2750, 'y' : -4100, 'z' : 0},
    {'name' : 'WATER CANYON', 'x' : 4100, 'y' : -5000, 'z' : 0},
    {'name' : 'KELLY', 'x' : 4500, 'y' : -200, 'z' : 0},
    {'name' : 'GUTIERREZ', 'x' : -4000, 'y' : 3000, 'z' : 0},
    {'name' : 'ABANDONED HOUSE', 'x' : -800, 'y' : 5000, 'z' : 0},
    {'name' : 'BOONDOCK', 'x' : 4500, 'y' : 8500, 'z' : 0},
    {'name' : 'WINDMILL', 'x' : 900, 'y' : 10000, 'z' : 0},
    {'name' : 'OUTOFSIGHT', 'x' : -4800, 'y' : 12500, 'z' : 0},
    # {'name' : '3 CM RADAR', 'x' : 0, 'y' : 0, 'z' : 1830},
    # {'name' : 'ETOT1', 'x' : 0, 'y' : 5100, 'z' : 1830},
    # {'name' : 'ETOT2', 'x' : 2900, 'y' : 400, 'z' : 1830},
    # {'name' : 'ETOT3', 'x' : 3100, 'y' : -6400, 'z' : 1830},
])

station_df

,name,x,y,z
0,STROZZI,-200,100,0
1,RECEIVING VANS,-2750,-4100,0
2,WATER CANYON,4100,-5000,0
3,KELLY,4500,-200,0
4,GUTIERREZ,-4000,3000,0
5,ABANDONED HOUSE,-800,5000,0
6,BOONDOCK,4500,8500,0
7,WINDMILL,900,10000,0
8,OUTOFSIGHT,-4800,12500,0


In [10]:
# create the arrays where the charge is located
Q = 3.7 # C

xgrid_q = np.arange(-10e3,11e3, 1e3) # in meters
ygrid_q = np.arange(-15e3, 16e3, 1e3)
z_q = np.array([3e3, 5e3, 7e3])

X, Y, Z = np.meshgrid(xgrid_q, ygrid_q, z_q)

Ri_dict = {}
mono_dE = {}

# calculate Ri vector for each station
for name in station_df['name']:
    x_obs = station_df.loc[station_df['name'] == name]['x'].values[0]
    y_obs = station_df.loc[station_df['name'] == name]['y'].values[0]
    
    Ri_x = X - x_obs
    Ri_y = Y - y_obs
    Ri_z = z_q

    Ri_mag = np.sqrt(Ri_x**2 + Ri_y**2 + Ri_z**2)

    dE = monopole_dE(Q, Ri_mag, Ri_z) # dE at each station


    mono_dE[name] = {'dE (kV/m)': dE/1000}

    Ri_dict[name] = {
        'Ri_x': Ri_x,
        'Ri_y': Ri_y,
        'Ri_z': Ri_z,
        'Ri_mag': Ri_mag
    }

In [11]:
mono_dE

{'STROZZI': {'dE (kV/m)': array([[[0.03282762, 0.05099421, 0.06461447],
          [0.03578295, 0.0553597 , 0.06976666],
          [0.03881568, 0.05980936, 0.07497121],
          ...,
          [0.03759739, 0.05802547, 0.07289025],
          [0.03458822, 0.05359845, 0.06769361],
          [0.03167937, 0.04929   , 0.06259036]],
  
         [[0.03767166, 0.05813435, 0.07301748],
          [0.04141571, 0.06360068, 0.07936964],
          [0.04531305, 0.06924454, 0.08585804],
          ...,
          [0.04374074, 0.06697318, 0.08325516],
          [0.03989571, 0.06138683, 0.07680524],
          [0.03623139, 0.05601954, 0.0705414 ]],
  
         [[0.04336183, 0.06642469, 0.08262494],
          [0.04813354, 0.07330062, 0.09047877],
          [0.05318177, 0.08050317, 0.09860036],
          ...,
          [0.05113523, 0.07759196, 0.09533031],
          [0.04618708, 0.07050397, 0.08729651],
          [0.04154673, 0.06379116, 0.07958977]],
  
         ...,
  
         [[0.04461376, 0.06823526, 0.0

### Q2: Inferring the monopolar discharge from the generated dE

You now have dE at all station locations, for all of the (assumed) monopolar discharges. Retreive the locations and charge amounts, pretending you didn't know them. This calculation will be slower, but only required a few seconds to completely run in my implementation.

Calculate and plot the error in the source location and charge magnitude at each position (a total of four error variables). Also plot the location of the measurement statios. There should be twelve plots - sets of four at each altitude. Is there any spatial variaibility in these error measures?

### Q3. Effect of measurement error on retrievals

Add a zero-mean distribution of [Gaussian errors](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html#numpy.random.Generator.normal) to the measured field changes. Use a standard deviation (aka RMS error) of 150 V/m, independently distributed at each station. 

Repeat the retrieval process and plot diagnostics for each error corrupted measurement. 

### Q4. Statistics of retrieval errors

Repeat Q3 for 30 error-corrupted E field measurements, so as to produce a large sample of statistics of the errors.

Calculate and plot the bias and standard deviation of position (each coordinate) and charge magnitude at each charge source location. This will be 24 plots, using all 30 realizations of the retrieval at each grid point.

### Q5: Visualizing the error stats as a chi^2 distribution
Calculate the reduced chi-squared value (in V/m) associated with each retrieved source, accounting for the degrees of freedom $\nu$, and normalizing by the RMS error you added above. You will have about 54000 values = (20x30x3 spatial)x(30 retrievals, one for each realization of error at each station). Create a histogram of these chi-squared values, and compare to the [analytic chi-squared distribution](https://en.wikipedia.org/wiki/Chi-squared_distribution) for that $\nu$, as in one of the panels in Thomas et al. (2004), Figure 9.